# Model Inference: Kaggle submission

Generates the final submission using the winning ensemble from `model_ensemble_fixed`:
**20% LightGBM + 80% N-BEATS** (best honest validation WMAE 2,210).

- LightGBM is retrained with the corrected preprocessing and predicts on the test set.
- N-BEATS test predictions come from `predictions/nn_test_preds.csv`.
- The blend is written to `submission.csv` in Kaggle format.

In [1]:
# Run from the project root so data/ and src/ work.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import yaml
import warnings
import numpy as np
import pandas as pd

from src.features.preprocessing import merge_and_enrich, get_model_ready_data
from src.models.lightgbm_pipeline import create_lightgbm_pipeline

warnings.filterwarnings("ignore")
SPLIT_DATE = "2012-01-01"

# Winning weights from model_ensemble_fixed.ipynb.
W_LGB, W_NN = 0.2, 0.8

# Feature columns the preprocessor expects (match get_model_ready_data).
NUMERIC = ["Store", "Dept", "Temperature", "Fuel_Price_Delta", "CPI_Delta",
           "Unemployment_Delta", "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4",
           "MarkDown5", "Size", "WeekOfYear", "Is_Black_Friday", "Pre_Christmas", "IsHoliday"]
CATEGORICAL = ["Type"]

In [3]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
test_raw = pd.read_csv("data/test.csv")
print("test rows:", len(test_raw))

test rows: 115064


## Train LightGBM and predict on the test set

In [4]:
X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
    train_raw, stores, features, split_date=SPLIT_DATE
)

with open("configs/lightgbm_config.yaml") as f:
    lgb_params = yaml.safe_load(f)["model"]["params"]

lgb_pipe = create_lightgbm_pipeline(preprocessor, lgb_params)
lgb_pipe.fit(X_train, y_train)
print("LightGBM trained.")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006945 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2342
[LightGBM] [Info] Number of data points in the train set: 294132, number of used features: 19
[LightGBM] [Info] Start training from score 16105.306894
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
LightGBM trained.


In [5]:
def build_test_matrix(train_raw, test_raw, stores, features):
    # Combine train + test so the delta features continue correctly into the test weeks.
    test2 = test_raw.copy()
    test2["Weekly_Sales"] = np.nan
    combined = pd.concat([train_raw, test2], ignore_index=True)

    df = merge_and_enrich(combined, stores, features)
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["Year"] = df["Date"].dt.year
    df["Is_Black_Friday"] = ((df["Date"].dt.month == 11) & (df["Date"].dt.day >= 23) & (df["Date"].dt.day <= 29)).astype(int)
    df["Pre_Christmas"] = (df["WeekOfYear"] == 51).astype(int)
    df["IsHoliday"] = df["IsHoliday"].astype(int)
    df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
    df["Fuel_Price_Delta"] = df.groupby(["Store", "Dept"])['Fuel_Price'].diff().fillna(0)
    df["CPI_Delta"] = df.groupby(["Store", "Dept"])['CPI'].diff().fillna(0)
    df["Unemployment_Delta"] = df.groupby(["Store", "Dept"])['Unemployment'].diff().fillna(0)
    return df[df["Weekly_Sales"].isna()].copy()

test_rows = build_test_matrix(train_raw, test_raw, stores, features)
test_rows["lgb_pred"] = lgb_pipe.predict(test_rows[NUMERIC + CATEGORICAL])
print("test feature rows:", len(test_rows))

test feature rows: 115064


## Blend with N-BEATS test predictions and write the submission

In [6]:
sub = test_raw[["Store", "Dept", "Date"]].copy()
sub["ds"] = pd.to_datetime(sub["Date"])

# LightGBM test predictions, aligned by key.
lgb_map = test_rows[["Store", "Dept", "Date", "lgb_pred"]].copy()
lgb_map["ds"] = pd.to_datetime(lgb_map["Date"])
sub = sub.merge(lgb_map[["Store", "Dept", "ds", "lgb_pred"]], on=["Store", "Dept", "ds"], how="left")

# N-BEATS test predictions, aligned by key.
nn_test = pd.read_csv("predictions/nn_test_preds.csv")
nn_test["ds"] = pd.to_datetime(nn_test["Date"])
sub = sub.merge(nn_test[["Store", "Dept", "ds", "nn_pred"]], on=["Store", "Dept", "ds"], how="left")

sub["lgb_pred"] = sub["lgb_pred"].fillna(0)
sub["nn_pred"] = sub["nn_pred"].fillna(0)
sub["Weekly_Sales"] = np.clip(W_LGB * sub["lgb_pred"] + W_NN * sub["nn_pred"], 0, None)
sub["Id"] = sub["Store"].astype(str) + "_" + sub["Dept"].astype(str) + "_" + sub["Date"].astype(str)

submission = sub[["Id", "Weekly_Sales"]]
submission.to_csv("submission.csv", index=False)

print("rows:", len(submission), "| NaNs:", int(submission["Weekly_Sales"].isna().sum()),
      "| negatives:", int((submission["Weekly_Sales"] < 0).sum()))
submission.head()

rows: 115064 | NaNs: 0 | negatives: 0


,Id,Weekly_Sales
0,1_1_2012-11-02,28698.919755
1,1_1_2012-11-09,23971.847583
2,1_1_2012-11-16,25384.774729
3,1_1_2012-11-23,25774.982388
4,1_1_2012-11-30,27893.948816


## Notes

- Upload `submission.csv` to Kaggle. It is the fixed ensemble (20% LightGBM + 80%
  N-BEATS), which beat N-BEATS alone on validation (2,210 vs 2,275).
- N-BEATS alone (nn_test_preds) scored 4,143 on Kaggle; this blend should come in at or
  below that.